In [ ]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np

#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

2025-08-12 14:41:25.351946: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-12 14:41:25.356261: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-08-12 14:41:25.356274: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [ ]:

def from_ortho(inp:scm.ortho)->scm.Bounds:
    """
    Takes an ortho object and abstracts out its bounds.
    Aggregation function: will hang if ortho is not done fitting yet. 
    Requires that the ortho still have its training data 
    so we can extract things like "number of cells per cell-type" and "number of MPRA barcodes per cell"
    """
    
    ret=scm.Bounds()

    #get the min & max nb parameters across all models
    mins=[]
    maxes=[]
    
    #for each model class
    for var in ["by_cell_type_parameters","by_cre_parameters"]:
        ## nb min & max ##
        for key in getattr(inp,var).nb:
            current=getattr(inp,var).nb[key].result()
            maxes.append(float(current.max()))
            mins.append(float(current.min()))
        
        thetas=[]
        for key in getattr(inp,var).theta:
            current=getattr(inp,var).theta[key].result()
            thetas.append(current)
        
        ## theta means ##
        #we could munge the strings & use setattr but i think this is more readable
        if var=="by_cell_type_parameters":
            ret.by_cell_type_theta=np.mean(thetas)

    
    #min & max nb and add to the return object
    ret.min_mpra_umi=min(mins)
    ret.max_mpra_umi=max(maxes)

    ## Parameters from training data ##    
            

    #by_cre_theta:float
    #by_cell_type_theta:float
    

    return ret

def description_from_bounds(inp:scm.Bounds,
                            type="simple_spread",
                            nb_override=None):
    """
    Returns a primordial description dask dataframe

    and a hypothesis set describing the (unimplemented)
    
    Only type implemented for now is `simple_spread`
    - tiles CREs over the bounds range 
    other will be 
    - `override` : allowing you to specify exactly what nb param
    """
    #known before you start or "to be optimized":
    #  cells per cell-type is a fixed parameter
    #  barcodes per CRE is a fixed parameter
    
    #for each cell type, cell, decide how many MPRA barcodes are transfected.
    #A simple way to model this is poisson
    #MOI can be converted to poisson parameters
    #https://virology.ws/2011/01/13/multiplicity-of-infection/
    #observed shendure dat is overdispersed here, eh... could include an option for nb...
    #probably the way to do it is
    # - calculate fraction of cells with 0, 1, 2, 3... barcodes
    # - multiple fractions by total number of cells & round
    # - make a table with n=total_cells cell barcodes
    # - add number of barcodes in each cell as a column according to fraction (n_transfected).
    # - randomize row order
    # - assign cell_type based on proportions
    # - randomly sample n_transfected MPRA barcodes for each row, then convert to tall
    # - merge in CRE identity 

    #for a non-tfection reporter setup, extend to 'all-by-all' (all mpra BC by all ) filling in zeroes

    #how to assign cells to combinations of 
    pass

def simple_spread():
    """
    Takes 
    """
    pass

In [2]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='8GB')
client=Client(cluster)

In [5]:
xyz=scm.ortho.load(client,".","test_save")

In [6]:
b=from_ortho(xyz)

In [7]:
b

Bounds(metadata=None, min_mpra_umi=1.977178826507623, max_mpra_umi=221.90297054935553, by_cre_theta=None, by_cell_type_theta=3.2166202, num_cres=None, cells_per_cell_type=None, moi=None)

In [3]:
dat=scm.scMPRA_data.from_tsv("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres_v3.tsv")
dat.ortho_filter()
dat.set_negative_controls(["nobody","weak"])
dat.set_reference_cell("liver")
primordial=scm.ortho()
primordial.criss_cross(client,dat=dat)
primordial.extract_params(client)

scMPRAforge: INFO: Dropped 0 of 27 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [4]:
primordial.save(".","test_save")

3.2567084
3.396007
3.3175333
3.2008197
2.1678116
3.2843478
3.2969668
3.3159997


In [11]:
xyz.training_data.data["umis_mpra_bc"]

0         74
1        255
2        101
3         27
4        101
        ... 
41966      6
41967      0
41968      2
41969      4
41970      0
Name: umis_mpra_bc, Length: 41971, dtype: int64

In [21]:
min(mins)

2.0135518849371543

In [25]:
cluster.close()